# VAR 2026 Colab runner

Chạy từng cell theo thứ tự từ trên xuống (Runtime > Run all cũng được, nhưng lần đầu nên chạy tay từng cell để dễ phát hiện lỗi sớm).

**Quy trình:** kiểm tra GPU -> clone code -> mount Drive + cài môi trường -> chạy thử `bonsai` (scene nhỏ, có benchmark công khai để đối chiếu) -> nếu ổn thì chạy toàn bộ 7 scene -> kiểm tra kết quả -> xem thử vài ảnh render trước khi nộp.

**Trước khi chạy — chỉ cần chuẩn bị đúng 2 thứ:**
1. **Notebook này**: không cần tự upload file — mở trực tiếp trên Colab qua **File > Open notebook > tab GitHub**, dán URL repo (`https://github.com/viethoang2503/viettel-ai-race-bts-digital`), chọn `notebooks/colab_runner.ipynb`.
2. **Dataset**: upload toàn bộ thư mục `VAI_NVS_DATA_ROUND2/` (7 scene) lên Google Drive, đặt đúng đường dẫn `MyDrive/var2026/VAI_NVS_DATA_ROUND2/` (Bước 3 bên dưới sẽ tự link nó vào code qua symlink — không copy lại, không tốn thêm dung lượng).

**An toàn khi bị ngắt kết nối:** notebook này chạy lại được từ đầu bất cứ lúc nào mà không mất tiến độ — cell clone tự `git pull` để lấy code mới nhất nếu code đã có sẵn, và `real_train_fn` tự bỏ qua scene nào đã train xong (dựa trên checkpoint + fingerprint dữ liệu đã lưu trên Drive).

**2 điểm rủi ro chưa từng chạy thật trên GPU (chỉ xác nhận được khi chạy notebook này lần đầu):**
1. Cache CUDA extension trên Drive (cell "Cài môi trường") — nếu lần chạy thứ 2 vẫn thấy dòng "Building ... from source" thay vì "Restoring cached", báo lại để điều chỉnh `environment/setup_colab.sh`.
2. Việc load checkpoint để render (`gs_render_fn.py`) — nếu cell chạy pipeline báo lỗi liên quan `GaussianModel.restore` hoặc `OptimizationParams`, chụp lại traceback đầy đủ để báo lại.

## Bước 0 — Kiểm tra GPU runtime
Phải thấy tên GPU (vd Tesla T4/A100) ở dưới. Nếu báo lỗi "command not found" hoặc không thấy GPU nào: vào menu **Runtime > Change runtime type > Hardware accelerator > GPU**, chọn Save, rồi chạy lại cell này.

In [ ]:
!nvidia-smi

## Bước 1 — Clone code từ GitHub
Chỉ clone nếu chưa có; nếu đã có sẵn (chạy lại notebook trong cùng session Colab) thì tự `git pull` để luôn lấy code mới nhất — quan trọng nếu có fix mới được push lên trong lúc bạn đang test.

**Quan trọng — `git pull` là CHƯA ĐỦ nếu bạn đã chạy pipeline (Bước 5) ít nhất 1 lần trong session này:** Python cache code Python đã import vào bộ nhớ của kernel; `git pull` chỉ cập nhật file trên đĩa chứ không nạp lại code đang chạy. Nếu bạn gặp lỗi, đã `git pull`, mà chạy lại Bước 5 vẫn lỗi y hệt như cũ — vào **Runtime > Restart session** rồi chạy lại từ Bước 1. Không cần build lại CUDA extension hay tải lại LPIPS model, mọi thứ vẫn nằm trên đĩa/Drive, restart chỉ reset tiến trình Python.

In [ ]:
import os

REPO_URL = "https://github.com/viethoang2503/viettel-ai-race-bts-digital.git"
REPO_DIR = "/content/viettel-ai-race-bts-digital"

if not os.path.isdir(REPO_DIR):
    !git clone --recurse-submodules {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git submodule update --init --recursive

## Bước 2 — Mount Google Drive
Phải chạy trong MỘT cell Python riêng như thế này (không được gọi qua `!bash ...`) — `drive.mount()` cần chạy trực tiếp trong kernel của notebook mới hiện được popup xin quyền, gọi qua subprocess của lệnh `!bash` sẽ lỗi `AttributeError: 'NoneType' object has no attribute 'kernel'`.

Sẽ hiện popup xin quyền truy cập Drive — bấm cho phép.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Bước 3 — Cài môi trường + link dataset
Lần chạy đầu tiên sẽ build 2 CUDA extension từ source (**5-10 phút**, thấy dòng "Building ... from source"); từ lần thứ 2 trở đi (kể cả sau khi bị ngắt kết nối, mở lại) sẽ restore từ cache trên Drive và chạy trong vài giây (dòng "Restoring cached ...").

Cell này cũng tự tạo symlink `VAI_NVS_DATA_ROUND2 -> MyDrive/var2026/VAI_NVS_DATA_ROUND2` — nếu thấy dòng `WARNING: dataset not found`, nghĩa là bạn chưa upload dataset lên đúng đường dẫn đó trên Drive; upload xong thì chạy lại cell này. Nếu thấy `ERROR: Google Drive is not mounted`, quay lại Bước 2 chạy cell mount trước.

**Kiểm tra:** dòng cuối cùng của output phải là `CUDA available: True`. Nếu là `False`, quay lại Bước 0 kiểm tra runtime type.

In [ ]:
!bash environment/setup_colab.sh

## Bước 4 — Chọn phạm vi chạy
**Khuyến nghị: chạy `bonsai_only` trước.** `bonsai` là scene chuẩn Mip-NeRF360 công khai, nhỏ, có số liệu published để đối chiếu — dùng để xác nhận toàn bộ pipeline (train -> render -> tính điểm -> đóng gói) chạy đúng trước khi tốn GPU hour cho 5 scene BTS thật.

Sau khi `bonsai_only` chạy xong và điểm số hợp lý (không phải 0 hay lỗi), đổi `RUN_MODE` thành `"all_scenes"` và chạy lại từ Bước 5 để train toàn bộ 7 scene (train.py mặc định 30k iteration/lần train, mỗi scene train 2 lần — có thể mất vài giờ mỗi scene tuỳ GPU, nên chạy `all_scenes` sẽ mất khá nhiều thời gian).

In [ ]:
RUN_MODE = "bonsai_only"  # đổi thành "all_scenes" sau khi bonsai_only chạy ổn
ITERATIONS = 30000

assert RUN_MODE in ("bonsai_only", "all_scenes")

## Bước 5 — Chạy pipeline
Với mỗi scene: undistort (nếu cần, tự động cho 5 scene BTS thật) -> train 2 lần (eval + full data) -> render holdout để tính điểm -> render `test_poses.csv` thật -> đóng gói `submission.zip`. Toàn bộ output (checkpoint, log, ảnh render, zip) ghi thẳng ra Drive tại `OUTPUT_ROOT` bên dưới — không mất gì nếu Colab bị ngắt giữa chừng, chỉ cần chạy lại notebook từ đầu.

In [ ]:
import functools
from pathlib import Path

from src.common.config import load_scenes
from src.evaluation.compute_metrics import load_lpips_model
from src.orchestrator.run_pipeline import run_baseline_pipeline
from src.rendering.gs_render_fn import real_render_fn
from src.training.gs_train_fn import real_train_fn

OUTPUT_ROOT = Path("/content/drive/MyDrive/var2026/outputs")

all_scenes = load_scenes()
scenes = [s for s in all_scenes if s.name == "bonsai"] if RUN_MODE == "bonsai_only" else all_scenes
print(f"RUN_MODE={RUN_MODE} -> chạy {len(scenes)} scene: {[s.name for s in scenes]}")

train_fn = functools.partial(real_train_fn, iterations=ITERATIONS)

result = run_baseline_pipeline(
    scenes=scenes,
    train_fn=train_fn,
    render_fn=real_render_fn,
    lpips_model=load_lpips_model(),
    psnr_max=30.0,
    output_root=OUTPUT_ROOT,
)

## Bước 6 — Đọc kết quả
- `Per-scene scores`: điểm holdout tự đo (0-1, càng cao càng tốt) — với `bonsai`, so thử với số liệu Mip-NeRF360 published để xác nhận hàm tính điểm đúng.
- `Skipped scenes`: scene bị bỏ qua do lỗi data — nếu có, `submission_zip` sẽ luôn là `None` (fail-closed, cả bài nộp bị giữ lại chứ không nộp thiếu scene).
- `Validation problems`: submission.zip đã đóng gói nhưng có lỗi (sai kích thước ảnh, thiếu file...) — zip vẫn nằm trên Drive để debug nhưng không được coi là hợp lệ.
- `submission_zip`: đường dẫn file `submission.zip` trên Drive nếu mọi thứ hợp lệ — đây là file để nộp.

In [ ]:
print("Per-scene scores:")
for name, score in result.per_scene_scores.items():
    print(f"  {name}: {score:.4f}")

if result.skipped_scenes:
    print("\nSkipped scenes:")
    for name, problems in result.skipped_scenes.items():
        print(f"  {name}: {problems}")

if result.validation_problems:
    print("\nValidation problems (submission withheld):")
    for problem in result.validation_problems:
        print(f"  {problem}")

print(f"\nsubmission_zip: {result.submission_zip}")
print(f"Toàn bộ output (checkpoint/log/render) nằm tại: {OUTPUT_ROOT}")

## Bước 7 — Kiểm tra mắt thường trước khi nộp (khuyến nghị)
Điểm số tự động trên holdout không chắc phản ánh đúng chất lượng render trên `test_poses.csv` thật (test pose có thể extrapolate xa hơn holdout tự tạo). Xem thử vài ảnh render thật trước khi tin vào điểm số — **chỉ xem, không sửa ảnh** (sửa thủ công ảnh đầu ra bị cấm theo đề bài).

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for scene in scenes:
    render_dir = OUTPUT_ROOT / scene.name / "test_render"
    if not render_dir.exists():
        continue
    sample_paths = sorted(render_dir.iterdir())[:3]
    if not sample_paths:
        continue
    fig, axes = plt.subplots(1, len(sample_paths), figsize=(5 * len(sample_paths), 5))
    axes = [axes] if len(sample_paths) == 1 else axes
    for ax, path in zip(axes, sample_paths):
        ax.imshow(Image.open(path))
        ax.set_title(f"{scene.name}/{path.name}")
        ax.axis("off")
    plt.show()